# Предсказываем цену квартиры

В этом ноутбуке мы соберём всё, что уже пробовали на курсе: посмотрим на данные, подготовим признаки и обучим модель для прогноза `price_rub`. Примеры можно вспомнить в папке `lesson_3` репозитория курса, а общий маршрут проекта — в файле `TASK.md` в корне папки `final_project`.

Для вас уже подготовлены основные шаги и проверки. Места `TODO` ждут вашего решения: выбирайте подходящие методы, сравнивайте идеи и записывайте, что получилось. Если код остановится в месте TODO, всё в порядке — сначала нужно дописать этот шаг.

<details>
<summary>Если хочется вспомнить математику</summary>

Линейная модель делает прогноз так: $\hat y=b+\sum_j w_jx_j$.

Одна из знакомых нам метрик — $MAE=\frac{1}{n}\sum_i |y_i-\hat y_i|$. Она показывает среднюю абсолютную ошибку в рублях: чем меньше, тем точнее прогноз. RMSE сильнее реагирует на крупные ошибки. У R² бывают и отрицательные значения — если встретите такое, посмотрите, насколько хорошо модель справляется по сравнению с постоянным ответом.

</details>


### Как найти места для вашей работы

В ноутбуке есть пометки `TODO` — в коде они выглядят как `# TODO`. Попробуйте поиск по слову `TODO` через `Ctrl+F` (`Cmd+F` на macOS): так будет проще пройти все шаги и ничего не упустить. Перед завершением ещё раз просмотрите найденные места.

Код дописывайте в кодовых ячейках, а наблюдения и объяснения — в Markdown под строкой `Ответ:`.

В ноутбуке используются `assert` — это готовые проверки, которые помогают убедиться, что данные и результаты имеют ожидаемый вид. В самих проверках ничего дописывать не нужно: просто запускайте их вместе с остальными ячейками. Если проверка сообщает об ошибке, вернитесь к соответствующему шагу и проверьте свой код.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Откройте ноутбук из папки final_project."

In [ ]:
df = pd.read_csv(ROOT / "data" / "regression.csv")
TARGET = "price_rub"
SERVICE = ["listing_id", "group_id", "split"]
df.head()

## 1. Познакомимся с данными

Начнём со знакомых действий из анализа данных. Посмотрите на типы столбцов, пропуски и распределения, а затем запишите 2–3 наблюдения, которые показались вам интересными.

Здесь пригодятся `info()`, `isna().mean()`, `describe()`, `value_counts()`, `hist()` и `scatter()`. Примеры можно найти в папке `lesson_2` репозитория курса. Заполнение пропусков пока оставим для обработки внутри модели — её мы будем учить только на train.

Попробуйте также посмотреть, как связаны площадь и цена: какие закономерности вы замечаете?


In [ ]:
# TODO: добавьте свой EDA и несколько понятных графиков.
df.info()
df.isna().mean().sort_values(ascending=False)

**TODO:** Запишите 2–3 наблюдения по анализу данных. Какие особенности вы учтёте при подготовке модели?

Ответ:

## 2. Разделим данные на train и test

Для простоты используем две части: на `train` учим обработку и модель, на `test` проверяем результат. Разбиение уже подготовлено в `split`. Сохраним его: связанные объявления должны оставаться в одной части.

In [ ]:
assert set(df["split"]) == {"train", "test"}
assert df.groupby("group_id")["split"].nunique().max() == 1
train = df.loc[df["split"].eq("train")].copy()
test = df.loc[df["split"].eq("test")].copy()
assert not train.empty and not test.empty
assert set(train["group_id"]).isdisjoint(test["group_id"])
print({"train": len(train), "test": len(test)})

## 3. Подготовим признаки простыми шагами

Вспомните, как мы отдельно работали с числами и категориями. Здесь можно использовать знакомые `fillna()` и `pd.get_dummies()`. Это подсказки, а не обязательный рецепт: свои идеи тоже приветствуются.

В `prepare_features` допишите подготовку таблицы. При первом вызове на train сохраните нужные значения в словарь `preparation`: например, медианы и названия столбцов после кодирования. При вызове на test используйте тот же словарь. Новые категории не должны менять список столбцов: здесь поможет `reindex(..., fill_value=0)`.

Масштабирование необязательно. Если его используете, обучите scaler только на train и также сохраните его в `preparation`. Для приложения повторите те же шаги обработки с сохранёнными значениями. Описание полей находится в `README.md` в папке `final_project/data`, а примеры — в папках `lesson_2`, `lesson_3` и `lesson_4` репозитория курса.

In [ ]:
features = [
    "area", "rooms", "floor", "floors_total", "lift_count",
    "city", "house_type", "market_type",
]
assert features and len(features) == len(set(features))
assert set(features).issubset(df.columns)
forbidden = set(SERVICE + ["price_rub", "above_reference"])
assert not set(features) & forbidden, "Исключите цели и служебные поля."

In [ ]:
def prepare_features(frame, preparation=None):
    data = frame.copy()
    # TODO: разделите поля на числовые и категориальные.
    # Для чисел можно использовать fillna(), для категорий — pd.get_dummies().
    # Если preparation is None: вычислите медианы по train и сохраните их,
    # а также список полученных столбцов в словарь preparation.
    # Иначе: используйте сохранённые значения без повторного вычисления.
    # Для одинакового набора столбцов пригодится:
    # data.reindex(columns=preparation["columns"], fill_value=0)
    # Верните числовую таблицу и словарь: return data, preparation.
    raise NotImplementedError("Допишите подготовку признаков простыми шагами.")


X_train, preparation = prepare_features(train[features])
X_test, _ = prepare_features(test[features], preparation)
y_train, y_test = train[TARGET], test[TARGET]
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert list(X_train.columns) == list(X_test.columns)
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

## 4. Обучим первую модель

Для начала подойдут `LinearRegression` или `Ridge`. Выберите модель в месте TODO, обучите её на подготовленном train и сравните с простым прогнозом медианой. Ниже метрики считаются на test.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge


def regression_metrics(y_true, prediction):
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
        "R2": r2_score(y_true, prediction),
    }


def build_model():
    # TODO: верните выбранную модель, например LinearRegression().
    raise NotImplementedError("Выберите модель регрессии.")


baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
model = build_model()
model.fit(X_train, y_train)

pd.DataFrame({
    "median_baseline": regression_metrics(y_test, baseline.predict(X_test)),
    "your_model": regression_metrics(y_test, model.predict(X_test)),
}).T

## 5. Попробуем ваши идеи

Сравните одну и ту же модель без района и улицы, с районом, а затем с районом и улицей. Попробуйте ещё одну идею из задания или предложите свою. Каждое решение обучайте только на train, а метрики запишите на test.

Для нового набора полей измените `features` и заново выполните подготовку и обучение. Сохраните результаты всех вариантов в небольшой таблице. Даже если улучшения нет, расскажите, что заметили.

В нашем упрощённом проекте только train и test. Если по test вы выбираете самый удачный вариант, его метрика уже не является независимой оценкой — просто укажите это в выводах. Для сохранения ниже используется текущий `model` и соответствующие ему `features` и `preparation`.

In [ ]:
# TODO: здесь разместите эксперименты и таблицу test-метрик.
# Например: Ridge alpha, масштабирование, новые признаки.
# Если преобразуете цену, возвращайте прогноз в рубли до расчёта метрик.

**TODO:** Какие варианты вы сравнили и что получилось? Кратко опишите наблюдения по таблице метрик.

Ответ:

In [ ]:
# Сохраним текущую обученную модель и те же правила обработки.
# Если после экспериментов возвращаетесь к другой модели, верните также
# её features и preparation, затем заново подготовьте X_test.
best_model = model
best_features = list(features)
best_preparation = preparation
assert best_features and len(best_features) == len(set(best_features))
assert set(best_features).issubset(df.columns)
assert not set(best_features) & forbidden

## 6. Запишем результаты и сохраним решение

Посмотрите на несколько крупных ошибок на test и попробуйте понять их причины. В итоговую таблицу занесите метрики используемой модели.

Если пробовали логарифм цены, возвращайте прогноз в рубли и в ноутбуке, и в приложении. В сохраняемом словаре можно оставить описание нужного обратного преобразования.

In [ ]:
X_final_test, _ = prepare_features(test[best_features], best_preparation)
test_prediction = np.asarray(best_model.predict(X_final_test))
test_metrics = regression_metrics(y_test, test_prediction)
pd.Series(test_metrics)

In [ ]:
errors = test[["listing_id", TARGET]].copy()
errors["prediction_rub"] = test_prediction
errors["absolute_error"] = abs(errors[TARGET] - errors["prediction_rub"])
errors.sort_values("absolute_error", ascending=False).head()

**TODO:** Рассмотрите несколько крупных ошибок модели. Что вы заметили и чем можете объяснить такие прогнозы?

Ответ:

In [ ]:
model_path = ROOT / "models" / "price_model.joblib"
model_path.parent.mkdir(exist_ok=True)
joblib.dump({
    "model": best_model, "features": best_features, "preparation": best_preparation,
}, model_path)

## 7. Убедимся, что всё работает

Эти небольшие проверки помогут убедиться, что для каждой квартиры получен числовой прогноз и что после сохранения и загрузки ответы остались прежними. Здесь нет требований к конкретным коэффициентам, алгоритму или минимальному качеству — у вас может быть своё решение.

Если увидите отрицательные цены, попробуйте разобраться, для каких квартир это происходит, и расскажите об этом в выводах как об ограничении модели.


In [ ]:
assert test_prediction.shape == (len(X_final_test),)
assert np.isfinite(test_prediction).all(), "Есть NaN или бесконечные прогнозы."
assert all(np.isfinite(value) for value in test_metrics.values())
assert model_path.is_file()
loaded = joblib.load(model_path)
assert loaded["features"] == best_features
X_loaded, _ = prepare_features(test[loaded["features"]], loaded["preparation"])
reloaded_prediction = np.asarray(loaded["model"].predict(X_loaded))
assert np.allclose(test_prediction, reloaded_prediction), "После загрузки прогноз изменился."
print("Основные проверки пройдены.")

## 8. Расскажите, что у вас получилось

Как ваша модель справилась по сравнению с медианным прогнозом? Помогли ли район и улица? Какая идея оказалась полезной? Добавьте таблицу test-метрик всех вариантов и ваши наблюдения.

Перед завершением перезапустите ядро и выполните ноутбук сверху вниз. После проверок можно подключать модель к `app.py`, повторяя ту же подготовку данных. Пример интерфейса можно найти в отдельном открытом репозитории `streamlit-app` с уже развёрнутым приложением.

Ответ: